In [ ]:
# ============================================================
# CUADERNO 08 — FIGURAS Y TABLAS PARA EL PAPER
# Genera todas las figuras faltantes para ADCAIJ
# ============================================================

In [ ]:
# ── CELDA 1: Configuración ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import SGDOneClassSVM
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

ruta_procesados = '/content/drive/MyDrive/IDS2018_procesados'
ruta_resultados = '/content/drive/MyDrive/IDS2018_resultados'
ruta_figuras    = f"{ruta_resultados}/figuras"
os.makedirs(ruta_figuras, exist_ok=True)

# Cargar datos
df_train_full = pd.read_csv(
    f"{ruta_procesados}/dataset_train.csv")
df_test = pd.read_csv(
    f"{ruta_procesados}/dataset_test.csv")
df_labels = pd.read_csv(
    f"{ruta_procesados}/dataset_test_labels.csv")

df_train_full = df_train_full.select_dtypes(include=[np.number])
df_test       = df_test.select_dtypes(include=[np.number])

y_test_labels = df_labels.iloc[:, 0]
y_test_binary = (y_test_labels != 'Benign').astype(int)

n_features = df_train_full.shape[1]
n_train    = int(len(df_train_full) * 0.70)

# Cargar CSVs de resultados
df_if   = pd.read_csv(f"{ruta_resultados}/IF_resultados_detallados.csv")
df_ae   = pd.read_csv(f"{ruta_resultados}/AE_resultados_umbral_dinamico.csv")
df_svm  = pd.read_csv(f"{ruta_resultados}/OCSVM_resultados_detallados.csv")
df_lstm = pd.read_csv(f"{ruta_resultados}/LSTMAE_A100_resultados_detallados.csv")

print(f"✅ Configuracion lista")
print(f"   Features  : {n_features}")
print(f"   Test set  : {len(df_test):,} filas")

Mounted at /content/drive
✅ Configuracion lista
   Features  : 13
   Test set  : 4,967,041 filas


In [ ]:
# ── CELDA 2: Función para graficar matriz de confusión ────────
def plot_confusion_matrix(cm, titulo, ruta_guardar):
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, interpolation='nearest',
                   cmap=plt.cm.Blues, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Benigno\n(Pred)', 'Ataque\n(Pred)'],
                       fontsize=11)
    ax.set_yticklabels(['Benigno\n(Real)', 'Ataque\n(Real)'],
                       fontsize=11)
    for i in range(2):
        for j in range(2):
            color = 'white' if cm_norm[i,j] > 0.5 else 'black'
            ax.text(j, i,
                    f'{cm_norm[i,j]:.2f}\n({cm[i,j]:,})',
                    ha='center', va='center',
                    color=color, fontsize=11,
                    fontweight='bold')
    ax.set_title(titulo, fontsize=13,
                 fontweight='bold', pad=15)
    ax.set_ylabel('Valor Real', fontsize=11)
    ax.set_xlabel('Valor Predicho', fontsize=11)
    plt.tight_layout()
    plt.savefig(ruta_guardar, dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.close()
    tn, fp, fn, tp = cm.ravel()
    print(f"✅ {titulo}")
    print(f"   TP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}")
    print(f"   Precision={tp/(tp+fp):.4f} "
          f"Recall={tp/(tp+fn):.4f} "
          f"F1={2*tp/(2*tp+fp+fn):.4f}")

print("✅ Funcion definida")

✅ Funcion definida


In [ ]:
# ── CELDA 3: Matriz de confusión — Isolation Forest ──────────
print("="*60)
print("MATRIZ DE CONFUSION — ISOLATION FOREST")
print("="*60)

np.random.seed(42)
idx_train = np.random.choice(
    len(df_train_full), size=n_train, replace=False)
X_train = df_train_full.iloc[idx_train].values
X_test  = df_test.values

modelo_if = IsolationForest(
    n_estimators=100, contamination=0.1,
    max_samples=min(256, len(X_train)),
    max_features=1, random_state=42, n_jobs=-1)
modelo_if.fit(X_train)

y_pred = (modelo_if.predict(X_test) == -1).astype(int)
cm = confusion_matrix(y_test_binary, y_pred)

plot_confusion_matrix(
    cm,
    "Isolation Forest — Matriz de Confusión",
    f"{ruta_figuras}/IF_confusion_matrix.png"
)

MATRIZ DE CONFUSION — ISOLATION FOREST
✅ Isolation Forest — Matriz de Confusión
   TP=188,999 FP=380,558 FN=972,272 TN=3,425,212
   Precision=0.3318 Recall=0.1628 F1=0.2184


In [ ]:
# ── CELDA 4: Matriz de confusión — One-Class SVM ─────────────
print("="*60)
print("MATRIZ DE CONFUSION — ONE-CLASS SVM")
print("="*60)

np.random.seed(42)
idx_svm = np.random.choice(
    len(df_train_full), size=50000, replace=False)
X_train_svm = df_train_full.iloc[idx_svm].values.astype(
    np.float32)

modelo_svm = make_pipeline(
    Nystroem(kernel='rbf', n_components=100,
             random_state=42),
    SGDOneClassSVM(nu=0.05, random_state=42,
                   max_iter=1000, tol=1e-3)
)
modelo_svm.fit(X_train_svm)

y_pred_svm = (
    modelo_svm.predict(X_test) == -1).astype(int)
cm = confusion_matrix(y_test_binary, y_pred_svm)

plot_confusion_matrix(
    cm,
    "One-Class SVM — Matriz de Confusión",
    f"{ruta_figuras}/OCSVM_confusion_matrix.png"
)

MATRIZ DE CONFUSION — ONE-CLASS SVM
✅ One-Class SVM — Matriz de Confusión
   TP=7,811 FP=162,317 FN=1,153,460 TN=3,643,453
   Precision=0.0459 Recall=0.0067 F1=0.0117


In [ ]:
# ── CELDA 5 FINAL: detección con lógica condicional ──────────
fig, ax = plt.subplots(figsize=(13, 8))
x = np.arange(4)
width = 0.35

bars1 = ax.bar(x - width/2, f1_medias, width,
               yerr=f1_stds, capsize=4,
               label='F1-Score', color='#2E86AB',
               alpha=0.85, error_kw={'linewidth':1.5,
                                     'zorder':5})
bars2 = ax.bar(x + width/2, auc_medias, width,
               yerr=auc_stds, capsize=4,
               label='AUC-ROC', color='#A23B72',
               alpha=0.85, error_kw={'linewidth':1.5,
                                     'zorder':5})

ax.set_ylabel('Valor', fontsize=13)
ax.set_title('Comparación de Calidad de Detección\n'
             'F1-Score y AUC-ROC por Modelo '
             '(media ± desv. estándar)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(modelos_nombres, fontsize=12)

max_con_error = max(
    max(v+s for v,s in zip(f1_medias, f1_stds)),
    max(v+s for v,s in zip(auc_medias, auc_stds))
)
ax.set_ylim(0, max_con_error * 1.12)
ax.legend(fontsize=12, loc='upper left')
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

UMBRAL = 0.15  # barras menores a este valor usan etiqueta encima

for bar, val, std in zip(bars1, f1_medias, f1_stds):
    if val >= UMBRAL:
        ax.text(bar.get_x() + bar.get_width()/2,
                val * 0.5,
                f'{val:.3f}',
                ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white')
    else:
        ax.text(bar.get_x() + bar.get_width()/2,
                val + std + 0.03,
                f'{val:.3f}',
                ha='center', va='bottom',
                fontsize=10, fontweight='bold',
                color='#1a5276')

for bar, val, std in zip(bars2, auc_medias, auc_stds):
    if val >= UMBRAL:
        ax.text(bar.get_x() + bar.get_width()/2,
                val * 0.5,
                f'{val:.3f}',
                ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white')
    else:
        ax.text(bar.get_x() + bar.get_width()/2,
                val + std + 0.03,
                f'{val:.3f}',
                ha='center', va='bottom',
                fontsize=10, fontweight='bold',
                color='#6c1a4a')

plt.tight_layout()
plt.savefig(f"{ruta_figuras}/comparacion_deteccion.png",
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("✅ comparacion_deteccion.png guardada")

✅ comparacion_deteccion.png guardada


In [ ]:
# ── CELDA 6 FINAL: costo con lógica condicional ──────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
colores = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

# Umbral: si la barra es menor al 15% del máximo
# → etiqueta encima en negro
UMBRAL_PORCENT = 0.15

# ── Subplot 1: Tiempo ────────────────────────────────────────
bars1 = ax1.bar(modelos_nombres, tiempos,
                yerr=tiempos_std, capsize=4,
                color=colores, alpha=0.85,
                error_kw={'linewidth':1.5, 'zorder':5})
ax1.set_ylabel('Segundos', fontsize=13)
ax1.set_title('Tiempo de Entrenamiento\n'
              '(media ± desv. estándar)',
              fontsize=13, fontweight='bold')
ax1.yaxis.grid(True, alpha=0.3)
ax1.set_axisbelow(True)

max_t = max(tiempos)
max_con_error_t = max(t+s for t,s in zip(tiempos, tiempos_std))
ax1.set_ylim(0, max_con_error_t * 1.15)

for bar, val, std in zip(bars1, tiempos, tiempos_std):
    if val >= max_t * UMBRAL_PORCENT:
        # Barra grande — etiqueta dentro en blanco
        ax1.text(bar.get_x() + bar.get_width()/2,
                 val * 0.5,
                 f'{val:,.1f}s',
                 ha='center', va='center',
                 fontsize=11, fontweight='bold',
                 color='white')
    else:
        # Barra pequeña — etiqueta encima en negro
        ax1.text(bar.get_x() + bar.get_width()/2,
                 val + std + max_t * 0.03,
                 f'{val:,.1f}s',
                 ha='center', va='bottom',
                 fontsize=11, fontweight='bold',
                 color='black')

# ── Subplot 2: Velocidad ──────────────────────────────────────
bars2 = ax2.bar(modelos_nombres, velocidades,
                color=colores, alpha=0.85)
ax2.set_ylabel('Muestras por segundo', fontsize=13)
ax2.set_title('Velocidad de Inferencia\n'
              '(muestras/segundo)',
              fontsize=13, fontweight='bold')
ax2.yaxis.grid(True, alpha=0.3)
ax2.set_axisbelow(True)

max_v = max(velocidades)
ax2.set_ylim(0, max_v * 1.15)

for bar, val in zip(bars2, velocidades):
    if val >= max_v * UMBRAL_PORCENT:
        ax2.text(bar.get_x() + bar.get_width()/2,
                 val * 0.5,
                 f'{val:,.0f}',
                 ha='center', va='center',
                 fontsize=11, fontweight='bold',
                 color='white')
    else:
        ax2.text(bar.get_x() + bar.get_width()/2,
                 val + max_v * 0.02,
                 f'{val:,.0f}',
                 ha='center', va='bottom',
                 fontsize=11, fontweight='bold',
                 color='black')

plt.suptitle(
    'Comparación de Costo Computacional — Variable Independiente',
    fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{ruta_figuras}/comparacion_costo.png",
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("✅ comparacion_costo.png guardada")

✅ comparacion_costo.png guardada


In [ ]:
# ── CELDA 7: Curvas de convergencia AE y LSTM-AE ─────────────
print("="*60)
print("CURVAS DE CONVERGENCIA")
print("="*60)

for nombre, archivo, loss_label in [
    ("Autoencoder", "AE_historial_convergencia.json", "MSE"),
    ("LSTM Autoencoder",
     "LSTMAE_historial_convergencia.json", "MAE")
]:
    ruta_json = f"{ruta_resultados}/{archivo}"
    if not os.path.exists(ruta_json):
        print(f"⚠️  {archivo} no encontrado — omitiendo")
        continue

    with open(ruta_json, 'r') as f:
        hist = json.load(f)

    epocas   = hist['epocas']
    loss     = hist['loss']
    val_loss = hist['val_loss']

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epocas, loss,
            label=f'Training loss ({loss_label})',
            color='#2E86AB', linewidth=2)
    ax.plot(epocas, val_loss,
            label=f'Validation loss ({loss_label})',
            color='#A23B72', linewidth=2,
            linestyle='--')
    ax.set_xlabel('Época', fontsize=12)
    ax.set_ylabel(f'Loss ({loss_label})', fontsize=12)
    ax.set_title(
        f'{nombre} — Curva de Convergencia\n'
        f'Training vs Validation Loss',
        fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

    # Marcar época de early stopping
    ax.axvline(x=len(epocas),
               color='gray', linestyle=':',
               alpha=0.7, label='Early stopping')
    ax.annotate(f'Early stopping\nÉpoca {len(epocas)}',
                xy=(len(epocas), min(val_loss)),
                xytext=(len(epocas)-3,
                        max(loss)*0.6),
                fontsize=9, color='gray',
                arrowprops=dict(arrowstyle='->',
                                color='gray'))

    plt.tight_layout()
    nombre_archivo = nombre.replace(' ', '_')
    ruta_conv = (f"{ruta_figuras}/"
                 f"{nombre_archivo}_convergencia.png")
    plt.savefig(ruta_conv, dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✅ Curva {nombre} guardada")
    print(f"   Epocas: {len(epocas)} | "
          f"Loss final: {loss[-1]:.6f} | "
          f"Val loss final: {val_loss[-1]:.6f}")

CURVAS DE CONVERGENCIA
✅ Curva Autoencoder guardada
   Epocas: 21 | Loss final: 0.000046 | Val loss final: 0.000032
✅ Curva LSTM Autoencoder guardada
   Epocas: 28 | Loss final: 0.000143 | Val loss final: 0.000225


In [ ]:
# ── CELDA 8: Resumen final de todas las figuras ──────────────
print("="*60)
print("FIGURAS GENERADAS")
print("="*60)

figuras_esperadas = {
    "IF_confusion_matrix.png":
        "Matriz confusion Isolation Forest",
    "OCSVM_confusion_matrix.png":
        "Matriz confusion One-Class SVM",
    "AE_confusion_matrix.png":
        "Matriz confusion Autoencoder",
    "LSTMAE_confusion_matrix.png":
        "Matriz confusion LSTM Autoencoder",
    "comparacion_deteccion.png":
        "Grafico F1-Score y AUC-ROC comparativo",
    "comparacion_costo.png":
        "Grafico costo computacional comparativo",
    "Autoencoder_convergencia.png":
        "Curva convergencia Autoencoder",
    "LSTM_Autoencoder_convergencia.png":
        "Curva convergencia LSTM Autoencoder",
}

todas_ok = True
for archivo, descripcion in figuras_esperadas.items():
    ruta = f"{ruta_figuras}/{archivo}"
    if os.path.exists(ruta):
        size = os.path.getsize(ruta) / 1024
        print(f"✅ {descripcion}")
        print(f"   {archivo} — {size:.1f} KB")
    else:
        print(f"❌ {descripcion} — NO ENCONTRADA")
        todas_ok = False

print("\n" + "="*60)
if todas_ok:
    print("✅ TODAS LAS FIGURAS GENERADAS CORRECTAMENTE")
else:
    print("⚠️  ALGUNAS FIGURAS FALTAN")
print(f"\nRuta: {ruta_figuras}")
print("="*60)

FIGURAS GENERADAS
✅ Matriz confusion Isolation Forest
   IF_confusion_matrix.png — 126.5 KB
✅ Matriz confusion One-Class SVM
   OCSVM_confusion_matrix.png — 126.8 KB
❌ Matriz confusion Autoencoder — NO ENCONTRADA
✅ Matriz confusion LSTM Autoencoder
   LSTMAE_confusion_matrix.png — 129.3 KB
✅ Grafico F1-Score y AUC-ROC comparativo
   comparacion_deteccion.png — 156.7 KB
✅ Grafico costo computacional comparativo
   comparacion_costo.png — 230.7 KB
✅ Curva convergencia Autoencoder
   Autoencoder_convergencia.png — 163.1 KB
✅ Curva convergencia LSTM Autoencoder
   LSTM_Autoencoder_convergencia.png — 171.0 KB

⚠️  ALGUNAS FIGURAS FALTAN

Ruta: /content/drive/MyDrive/IDS2018_resultados/figuras
